Prepared HTTP V2 extract. Exact original ZIP bytes stream once through the reviewed Range code into owned /tmp. Globally pinned receipts retain original provenance, including fresh-DICOM source5. Owner-approved MRI pilot, full-route forecast and fresh quota gate are required. Preparation only; no GPU job or signed URL has been created.

In [ ]:
import time
BOOTSTRAP_MONOTONIC=time.monotonic()
print('{"event":"bootstrap_start","diagnostic_schema":1}',flush=True)
import json,os,time,traceback
BOOTSTRAP_DIR='/kaggle/working'
BOOTSTRAP_STAGE='notebook_start'
os.makedirs(BOOTSTRAP_DIR,exist_ok=True)
def bootstrap_checkpoint(stage):
    global BOOTSTRAP_STAGE
    BOOTSTRAP_STAGE=stage
    event={'diagnostic_schema':1,'event':'bootstrap_checkpoint','stage':stage,'unix_seconds':time.time()}
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_progress.json'),'w') as stream:
        json.dump(event,stream,indent=2)
    with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_events.jsonl'),'a') as stream:
        stream.write(json.dumps(event)+'\n')
    print(json.dumps(event),flush=True)
def bootstrap_save_failure(error):
    event={'diagnostic_schema':1,'event':'bootstrap_failure','stage':BOOTSTRAP_STAGE,
           'error_type':type(error).__name__,'error':str(error),'traceback':traceback.format_exc(),
           'unix_seconds':time.time()}
    try:
        with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_failure.json'),'w') as stream:
            json.dump(event,stream,indent=2)
    except BaseException as save_error:
        print(json.dumps({'event':'bootstrap_failure_save_error','error_type':type(save_error).__name__}),flush=True)
    print(json.dumps(event),flush=True)
bootstrap_checkpoint('notebook_start')
with open(os.path.join(BOOTSTRAP_DIR,'bootstrap_start.json'),'w') as stream:
    json.dump({'diagnostic_schema':1,'event':'bootstrap_start','unix_seconds':time.time()},stream,indent=2)


In [ ]:
from pathlib import Path
import json,os,sys,time
GUARD_SOURCE='"""Own one launcher process group and scratch root; redact all child output."""\nimport json\nimport os\nimport shutil\nimport signal\nimport tempfile\nimport time\nfrom pathlib import Path\n\n\ndef guarded_launcher(prepare, deadline, output, scratch_parent="/tmp", env=None):\n    started = time.monotonic()\n    scratch, child_pid, child_status = None, None, None\n    report = {"complete": False, "route": "direct_http",\n              "account_credentials_given_to_child": False,\n              "raw_child_stdout_stderr_published": False,\n              "hard_process_group_deadline": True}\n    def stop_owned_group():\n        nonlocal child_status\n        if child_pid is not None:\n            try:\n                os.killpg(child_pid, signal.SIGKILL)\n                report["owned_process_group_terminated"] = True\n                if child_status is not None:\n                    report["owned_worker_group_outlived_launcher"] = True\n                    if report.get("complete"):\n                        report.update(complete=False, failure_code="orphaned_owned_worker")\n            except ProcessLookupError:\n                # A deadline can race the owned child before its first setsid.\n                # Never signal the inherited parent group; target this PID only.\n                if child_status is None:\n                    try:os.kill(child_pid,signal.SIGKILL)\n                    except ProcessLookupError:pass\n            if child_status is None:\n                _,child_status=os.waitpid(child_pid,0)\n    try:\n        parent = Path(scratch_parent)\n        parent.mkdir(parents=True, exist_ok=True)\n        scratch = Path(tempfile.mkdtemp(prefix="rsna-http-cache0-", dir=parent))\n        if time.monotonic()>=deadline:raise TimeoutError()\n        child_env = dict(os.environ if env is None else env)\n        for field in ("KAGGLE_API_TOKEN", "KAGGLE_KEY", "KAGGLE_USERNAME"):\n            child_env.pop(field, None)\n        child_pid=os.fork()\n        if child_pid==0:\n            # Every preparation/input read/copy/compile runs under the parent\'s\n            # watchdog. The scientific command then replaces this group leader.\n            try:\n                os.setsid()\n                own_temp=scratch/\'temporary\';own_temp.mkdir()\n                child_env.update(TMPDIR=str(own_temp),TMP=str(own_temp),TEMP=str(own_temp),\n                                 RSNA_HTTP_OWNED_TMP=str(own_temp))\n                os.environ.clear();os.environ.update(child_env);tempfile.tempdir=str(own_temp)\n                with (scratch/\'child_capture.log\').open(\'wb\') as capture:\n                    os.dup2(capture.fileno(),1);os.dup2(capture.fileno(),2)\n                cmd,additions=prepare(scratch)\n                child_env.update(additions)\n                for field in (\'KAGGLE_API_TOKEN\',\'KAGGLE_KEY\',\'KAGGLE_USERNAME\'):child_env.pop(field,None)\n                (scratch/\'preparation_complete\').write_text(\'complete\')\n                os.execvpe(cmd[0],cmd,child_env)\n            except BaseException:\n                os._exit(125)\n        while True:\n            waited,status=os.waitpid(child_pid,os.WNOHANG)\n            if waited:\n                child_status=status;break\n            remaining=deadline-time.monotonic()\n            if remaining<=0:raise TimeoutError()\n            time.sleep(min(.01,remaining))\n        if time.monotonic()>=deadline:raise TimeoutError()\n        returncode=os.waitstatus_to_exitcode(child_status)\n        report.update(child_exit_code=returncode, complete=returncode == 0)\n        if returncode:\n            report["failure_code"] = \'redacted_owned_child_failure\' if (scratch/\'preparation_complete\').exists() else \'redacted_parent_preparation_failure\'\n    except TimeoutError:\n        stop_owned_group()\n        report.update(complete=False, failure_code="hard_walltime_budget")\n    except BaseException as error:\n        report.update(complete=False, failure_code="redacted_parent_preparation_failure",\n                      exception_type=type(error).__name__)\n    finally:\n        stop_owned_group()\n        if scratch is not None:\n            try:\n                shutil.rmtree(scratch)\n            except OSError as error:\n                report.update(complete=False, cleanup_exception_type=type(error).__name__)\n        report["owned_tmp_removed"] = scratch is None or not scratch.exists()\n        report["seconds"] = time.monotonic() - started\n        output = Path(output)\n        output.parent.mkdir(parents=True, exist_ok=True)\n        temporary = output.with_suffix(output.suffix + ".tmp")\n        temporary.write_text(json.dumps(report, indent=2))\n        temporary.replace(output)\n    return report\n'
CHILD_SOURCE='import time\nimport os\nBOOTSTRAP_MONOTONIC=float(os.environ[\'RSNA_HTTP_PILOT_START\'])\nprint(\'{"event":"bootstrap_start","diagnostic_schema":1}\',flush=True)\nimport json,os,time,traceback\nBOOTSTRAP_DIR=\'/kaggle/working\'\nBOOTSTRAP_STAGE=\'notebook_start\'\nos.makedirs(BOOTSTRAP_DIR,exist_ok=True)\ndef bootstrap_checkpoint(stage):\n    global BOOTSTRAP_STAGE\n    BOOTSTRAP_STAGE=stage\n    event={\'diagnostic_schema\':1,\'event\':\'bootstrap_checkpoint\',\'stage\':stage,\'unix_seconds\':time.time()}\n    with open(os.path.join(BOOTSTRAP_DIR,\'bootstrap_progress.json\'),\'w\') as stream:\n        json.dump(event,stream,indent=2)\n    with open(os.path.join(BOOTSTRAP_DIR,\'bootstrap_events.jsonl\'),\'a\') as stream:\n        stream.write(json.dumps(event)+\'\\n\')\n    print(json.dumps(event),flush=True)\ndef bootstrap_save_failure(error):\n    event={\'diagnostic_schema\':1,\'event\':\'bootstrap_failure\',\'stage\':BOOTSTRAP_STAGE,\n           \'error_type\':type(error).__name__,\'error\':str(error),\'traceback\':traceback.format_exc(),\n           \'unix_seconds\':time.time()}\n    try:\n        with open(os.path.join(BOOTSTRAP_DIR,\'bootstrap_failure.json\'),\'w\') as stream:\n            json.dump(event,stream,indent=2)\n    except BaseException as save_error:\n        print(json.dumps({\'event\':\'bootstrap_failure_save_error\',\'error_type\':type(save_error).__name__}),flush=True)\n    print(json.dumps(event),flush=True)\nbootstrap_checkpoint(\'notebook_start\')\nwith open(os.path.join(BOOTSTRAP_DIR,\'bootstrap_start.json\'),\'w\') as stream:\n    json.dump({\'diagnostic_schema\':1,\'event\':\'bootstrap_start\',\'unix_seconds\':time.time()},stream,indent=2)\nimport importlib.util,json,os,time\nfrom pathlib import Path\nHTTP_SPEC={\'shard\': 2, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-2-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-2-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10302281809, \'sha256\': \'c7703128578cb7eac8141e309f0ada35a3eb616ff2ec6d19a611160a7ef5cc38\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_2\', \'fingerprint\': {\'plan_sha256\': \'32fc0fbdd03759d1a6b7167ace7ff7659b550b78e4cfc038f370e4dfb7509793\', \'metadata_sha256\': \'88d3dc06b9bb788d0f0ed23cf95e4eb8a502c378042d9bb8fe09f9e198fd3d7b\', \'summary_sha256\': \'a7041fbd39b0f348ca50480b8598a72917f2a06af8787c09cf2c5af9d2b5a5e1\', \'archive_sha256\': \'c7703128578cb7eac8141e309f0ada35a3eb616ff2ec6d19a611160a7ef5cc38\', \'manifest_sha256\': \'8ac9904d5a501f3103581755d8c16c6c542d7f04b1e7231689c8096a8f0cfc52\', \'source_summary_sha256\': \'e30d65b481432d2c7182b0b618b56e88c9dfbc11e95701c39ef083a15a28fa9a\', \'repack_receipt_sha256\': \'be02689ae9e44e455795d771c65ad6851d7c31ed0ce7b7e8c5a3540bc9ad2c92\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 646487, \'sha256\': \'88d3dc06b9bb788d0f0ed23cf95e4eb8a502c378042d9bb8fe09f9e198fd3d7b\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1007, \'sha256\': \'a7041fbd39b0f348ca50480b8598a72917f2a06af8787c09cf2c5af9d2b5a5e1\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1703817, \'sha256\': \'8ac9904d5a501f3103581755d8c16c6c542d7f04b1e7231689c8096a8f0cfc52\'}, {\'path\': \'port/plan.csv\', \'bytes\': 541554, \'sha256\': \'32fc0fbdd03759d1a6b7167ace7ff7659b550b78e4cfc038f370e4dfb7509793\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 691, \'sha256\': \'be02689ae9e44e455795d771c65ad6851d7c31ed0ce7b7e8c5a3540bc9ad2c92\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 402, \'sha256\': \'e30d65b481432d2c7182b0b618b56e88c9dfbc11e95701c39ef083a15a28fa9a\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1236, \'expected_images\': 12580}\nHTTP_SOURCE_INDEX=[{\'shard\': 0, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-0-zip\', \'origin\': {\'kind\': \'dataset\', \'identifier\': \'alanchoo/rsna-knee-cache-0-bin-v1\', \'version_number\': 1, \'filename\': \'cache384_train.zip.bin\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10241860213, \'sha256\': \'31b3dd3934e79931fa78195315bef1b58d8432bae841b7920c6d4497262e764c\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_0\', \'fingerprint\': {\'plan_sha256\': \'ddd0c339d29d875e2aaeb9e38a15fd6804e1ba6aaebb8f4977aadc0cae113db9\', \'metadata_sha256\': \'ab5fb8a7714346b53d4038c6fc41f2f0555e38c384b30f8bfaf0dd77d76b6e1b\', \'summary_sha256\': \'2982a09847900f9c84e94ca3e492a52353c6d219ac51b28a7957c852fee248ae\', \'archive_sha256\': \'31b3dd3934e79931fa78195315bef1b58d8432bae841b7920c6d4497262e764c\', \'manifest_sha256\': \'d1597336ef00be65c862238e0993a23311ef8279ee220c1efc024890bc2c3cd0\', \'source_summary_sha256\': \'650e44a5c7ed2bb72082c25ea48059ebcef2655290757a5a0713be5940e9eded\', \'repack_receipt_sha256\': \'e93119c90f953ec2c83b15210350b4baa99255e6d639f4d8247fb1cef1120eb2\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 632893, \'sha256\': \'ab5fb8a7714346b53d4038c6fc41f2f0555e38c384b30f8bfaf0dd77d76b6e1b\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1011, \'sha256\': \'2982a09847900f9c84e94ca3e492a52353c6d219ac51b28a7957c852fee248ae\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1667720, \'sha256\': \'d1597336ef00be65c862238e0993a23311ef8279ee220c1efc024890bc2c3cd0\'}, {\'path\': \'port/plan.csv\', \'bytes\': 530106, \'sha256\': \'ddd0c339d29d875e2aaeb9e38a15fd6804e1ba6aaebb8f4977aadc0cae113db9\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 691, \'sha256\': \'e93119c90f953ec2c83b15210350b4baa99255e6d639f4d8247fb1cef1120eb2\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 406, \'sha256\': \'650e44a5c7ed2bb72082c25ea48059ebcef2655290757a5a0713be5940e9eded\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1204, \'expected_images\': 12240}, {\'shard\': 1, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-1-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-1-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10275094284, \'sha256\': \'b7d73673925bba6e43908b54c5d27a7b027383853d4046028517b46bd65755fe\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_1\', \'fingerprint\': {\'plan_sha256\': \'f011f6bc9e4258a950240610c4c0c9414bdc8555f5ce0c721ae620254f37f80c\', \'metadata_sha256\': \'e5ffddbd03f620bf6bf4519412b501602369181991ec5c587c087a0629f27063\', \'summary_sha256\': \'fd5c59300123e5914ac67a071b4a2754bc5b02c0fb8fdb764353dd47a9454b34\', \'archive_sha256\': \'b7d73673925bba6e43908b54c5d27a7b027383853d4046028517b46bd65755fe\', \'manifest_sha256\': \'3cea62d88455a6f0d6037f870d4e2f129e4c06f0e8c841f0e36144f66ade1fb8\', \'source_summary_sha256\': \'c6f8487ddcdcb53dcbd5c6254c361fb66fd3caadacdfd8cdcae95a30073a1f2a\', \'repack_receipt_sha256\': \'609b4b5c2f24818e1dd42f24607871d91fb18fdd531ae721beb31460e6a2b5b3\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 643878, \'sha256\': \'e5ffddbd03f620bf6bf4519412b501602369181991ec5c587c087a0629f27063\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1007, \'sha256\': \'fd5c59300123e5914ac67a071b4a2754bc5b02c0fb8fdb764353dd47a9454b34\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1696682, \'sha256\': \'3cea62d88455a6f0d6037f870d4e2f129e4c06f0e8c841f0e36144f66ade1fb8\'}, {\'path\': \'port/plan.csv\', \'bytes\': 539235, \'sha256\': \'f011f6bc9e4258a950240610c4c0c9414bdc8555f5ce0c721ae620254f37f80c\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 687, \'sha256\': \'609b4b5c2f24818e1dd42f24607871d91fb18fdd531ae721beb31460e6a2b5b3\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 402, \'sha256\': \'c6f8487ddcdcb53dcbd5c6254c361fb66fd3caadacdfd8cdcae95a30073a1f2a\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1231, \'expected_images\': 12528}, {\'shard\': 2, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-2-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-2-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10302281809, \'sha256\': \'c7703128578cb7eac8141e309f0ada35a3eb616ff2ec6d19a611160a7ef5cc38\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_2\', \'fingerprint\': {\'plan_sha256\': \'32fc0fbdd03759d1a6b7167ace7ff7659b550b78e4cfc038f370e4dfb7509793\', \'metadata_sha256\': \'88d3dc06b9bb788d0f0ed23cf95e4eb8a502c378042d9bb8fe09f9e198fd3d7b\', \'summary_sha256\': \'a7041fbd39b0f348ca50480b8598a72917f2a06af8787c09cf2c5af9d2b5a5e1\', \'archive_sha256\': \'c7703128578cb7eac8141e309f0ada35a3eb616ff2ec6d19a611160a7ef5cc38\', \'manifest_sha256\': \'8ac9904d5a501f3103581755d8c16c6c542d7f04b1e7231689c8096a8f0cfc52\', \'source_summary_sha256\': \'e30d65b481432d2c7182b0b618b56e88c9dfbc11e95701c39ef083a15a28fa9a\', \'repack_receipt_sha256\': \'be02689ae9e44e455795d771c65ad6851d7c31ed0ce7b7e8c5a3540bc9ad2c92\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 646487, \'sha256\': \'88d3dc06b9bb788d0f0ed23cf95e4eb8a502c378042d9bb8fe09f9e198fd3d7b\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1007, \'sha256\': \'a7041fbd39b0f348ca50480b8598a72917f2a06af8787c09cf2c5af9d2b5a5e1\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1703817, \'sha256\': \'8ac9904d5a501f3103581755d8c16c6c542d7f04b1e7231689c8096a8f0cfc52\'}, {\'path\': \'port/plan.csv\', \'bytes\': 541554, \'sha256\': \'32fc0fbdd03759d1a6b7167ace7ff7659b550b78e4cfc038f370e4dfb7509793\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 691, \'sha256\': \'be02689ae9e44e455795d771c65ad6851d7c31ed0ce7b7e8c5a3540bc9ad2c92\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 402, \'sha256\': \'e30d65b481432d2c7182b0b618b56e88c9dfbc11e95701c39ef083a15a28fa9a\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1236, \'expected_images\': 12580}, {\'shard\': 3, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-3-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-3-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10277511797, \'sha256\': \'b0f85db02d32467da7d433b4cfd6b10d71b15f9419d52119f19ef9abbf9cec3e\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_3\', \'fingerprint\': {\'plan_sha256\': \'6a11265070ec73995cfd9f61c63f506063bc39397605c4f053f05dc48b52b395\', \'metadata_sha256\': \'1fa77f37dca90b7954403ee4644594230b5ba73fe76bc52ef637bb6cc9d23be8\', \'summary_sha256\': \'532b7d09f048d4f05dc86b6bb689d72c6d2004cde9f5e15d0cd6256d2c785f21\', \'archive_sha256\': \'b0f85db02d32467da7d433b4cfd6b10d71b15f9419d52119f19ef9abbf9cec3e\', \'manifest_sha256\': \'57d66f07c984973dc987a6657f695764ebb53ff0c18283508c5a91af8902af49\', \'source_summary_sha256\': \'a982b5c47dea05d26aaefc967108b73c761fa2b1eb32193ad5de329efede8e89\', \'repack_receipt_sha256\': \'1216666293208ff5f807ca677a433053913d52a7f0876b19f057866219510614\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 639992, \'sha256\': \'1fa77f37dca90b7954403ee4644594230b5ba73fe76bc52ef637bb6cc9d23be8\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1011, \'sha256\': \'532b7d09f048d4f05dc86b6bb689d72c6d2004cde9f5e15d0cd6256d2c785f21\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1686718, \'sha256\': \'57d66f07c984973dc987a6657f695764ebb53ff0c18283508c5a91af8902af49\'}, {\'path\': \'port/plan.csv\', \'bytes\': 536147, \'sha256\': \'6a11265070ec73995cfd9f61c63f506063bc39397605c4f053f05dc48b52b395\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 687, \'sha256\': \'1216666293208ff5f807ca677a433053913d52a7f0876b19f057866219510614\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 406, \'sha256\': \'a982b5c47dea05d26aaefc967108b73c761fa2b1eb32193ad5de329efede8e89\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1213, \'expected_images\': 12332}, {\'shard\': 4, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-4-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-4-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10260139326, \'sha256\': \'369c83f8edbbac058a7f366d1a3a5794739531ba08f3f142cbe172dd56b1e146\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_4\', \'fingerprint\': {\'plan_sha256\': \'d363bf2bd578048d5b27199648e265fb5b3406a4538be4cc53b220fd568824d0\', \'metadata_sha256\': \'fe8867bd7eb6a5653fabe5000c849e4d2e4ddb446df9cef0213ea3fb9b84eecb\', \'summary_sha256\': \'a862d42e8547f3d54b84c216814ed9503ab86c2406a9a26bd43042cb183f99e3\', \'archive_sha256\': \'369c83f8edbbac058a7f366d1a3a5794739531ba08f3f142cbe172dd56b1e146\', \'manifest_sha256\': \'15c48b723c6aaa0f34b7311fa5d1f2d92d97c86ad69d02678245d5d5af7584bd\', \'source_summary_sha256\': \'f468addcaa1243ff647eb161a8f4ccaaf749ce62edce53d2de19b3be8ccb4bcf\', \'repack_receipt_sha256\': \'c67df6920c5f748d1a4e4054aeabc2a3e03b71944b21a24350af240733ffa075\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 639046, \'sha256\': \'fe8867bd7eb6a5653fabe5000c849e4d2e4ddb446df9cef0213ea3fb9b84eecb\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1011, \'sha256\': \'a862d42e8547f3d54b84c216814ed9503ab86c2406a9a26bd43042cb183f99e3\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1684342, \'sha256\': \'15c48b723c6aaa0f34b7311fa5d1f2d92d97c86ad69d02678245d5d5af7584bd\'}, {\'path\': \'port/plan.csv\', \'bytes\': 535327, \'sha256\': \'d363bf2bd578048d5b27199648e265fb5b3406a4538be4cc53b220fd568824d0\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 686, \'sha256\': \'c67df6920c5f748d1a4e4054aeabc2a3e03b71944b21a24350af240733ffa075\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 406, \'sha256\': \'f468addcaa1243ff647eb161a8f4ccaaf749ce62edce53d2de19b3be8ccb4bcf\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1216, \'expected_images\': 12364}, {\'shard\': 5, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-5-direct-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-5-direct-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 10364472100, \'sha256\': \'239160bd8c25f9bb7f2d94dadaca0abd57169cf4e86ef4c3f52027bcb779efff\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_5\', \'fingerprint\': {\'plan_sha256\': \'2f293f279783b8c2481c21d3af7f6441ebf0d50a37a86e132ad8d20bc980b597\', \'metadata_sha256\': \'85c14b30de0b3e2a591d0f59b563e9c3752215b2b4dc8793ce46971cabc60bc7\', \'summary_sha256\': \'5370ee29ff299a5bb85c4e3cf7bcc2dce23856616d6b56334416b5b3f90c10a2\', \'archive_sha256\': \'239160bd8c25f9bb7f2d94dadaca0abd57169cf4e86ef4c3f52027bcb779efff\', \'manifest_sha256\': \'f467f477478f7a3cf4fdcced39cf2e063579fabbe71d4955d586ff26b306d9fa\', \'source_summary_sha256\': \'1fd89c689f31313c9e6db1e63a466e445878e79d809f823c251924c22a0eb140\', \'direct_decode_receipt_sha256\': \'ba9d84bb6883554e1b69a3ce8426e9dc45b680fd76174e20224f0387c66e9e30\', \'direct_decode_metadata_sha256\': \'34745efdb75de3949d8ce56cf56599b52679f3fa7a8d8d4579f2759d417e9dae\', \'reference_summary_sha256\': \'35825df8a91fd866ed3df5adae73855d3ef0520062899b6d8600f687adbeaf9b\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 640755, \'sha256\': \'85c14b30de0b3e2a591d0f59b563e9c3752215b2b4dc8793ce46971cabc60bc7\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1155, \'sha256\': \'5370ee29ff299a5bb85c4e3cf7bcc2dce23856616d6b56334416b5b3f90c10a2\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1905469, \'sha256\': \'f467f477478f7a3cf4fdcced39cf2e063579fabbe71d4955d586ff26b306d9fa\'}, {\'path\': \'direct_decode_metadata.json\', \'bytes\': 1216591, \'sha256\': \'34745efdb75de3949d8ce56cf56599b52679f3fa7a8d8d4579f2759d417e9dae\'}, {\'path\': \'direct_decode_receipt.json\', \'bytes\': 1176, \'sha256\': \'ba9d84bb6883554e1b69a3ce8426e9dc45b680fd76174e20224f0387c66e9e30\'}, {\'path\': \'port/plan.csv\', \'bytes\': 536739, \'sha256\': \'2f293f279783b8c2481c21d3af7f6441ebf0d50a37a86e132ad8d20bc980b597\'}, {\'path\': \'reference_cache_summary.json\', \'bytes\': 402, \'sha256\': \'35825df8a91fd866ed3df5adae73855d3ef0520062899b6d8600f687adbeaf9b\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 604, \'sha256\': \'1fd89c689f31313c9e6db1e63a466e445878e79d809f823c251924c22a0eb140\'}], \'storage_kind\': \'competition_dicom\', \'original_npy_byte_identity_claim\': False, \'runtime_seconds\': 1228, \'expected_images\': 12496}, {\'shard\': 6, \'cache_kernel_id\': \'alanchoo/rsna-knee-cnxt-cache-6-zip\', \'origin\': {\'kind\': \'kernel\', \'identifier\': \'alanchoo/rsna-knee-cnxt-cache-6-zip\', \'version_number\': 1, \'filename\': \'cache384_train.zip\'}, \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': 8773191852, \'sha256\': \'b0196a1fb89272fa91aa16023477d745cc82414cdac7047657ca9e079ee7a4fa\'}, \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': \'shard_6\', \'fingerprint\': {\'plan_sha256\': \'b17605be3bcc36b00c7974c7084b952b8309bf08b77cc0d887c701cc63f107cd\', \'metadata_sha256\': \'f430ac2666fc458b6401a31acefc095c65c2467c03d6d7bdb3f0819ebe9454ee\', \'summary_sha256\': \'6a828ea6b59def42684a769413aa9f7017779a472977cb5fa04b03c9728c9c28\', \'archive_sha256\': \'b0196a1fb89272fa91aa16023477d745cc82414cdac7047657ca9e079ee7a4fa\', \'manifest_sha256\': \'aa3be67abdd715c74c597e3a02b98cda01d7b59d0ecf50fa465c15ce7433f325\', \'source_summary_sha256\': \'5505d293529518a60706f1873e68ce3d6c13ed5f1b25e37c0fdb7c86710f3da0\', \'repack_receipt_sha256\': \'0993f73b1c00646bec20e9edf89766098a06e55df56ce27c7854cc55fb15befb\'}, \'receipt_members\': [{\'path\': \'cache384_train_meta.csv\', \'bytes\': 549400, \'sha256\': \'f430ac2666fc458b6401a31acefc095c65c2467c03d6d7bdb3f0819ebe9454ee\'}, {\'path\': \'cache_summary.json\', \'bytes\': 1009, \'sha256\': \'6a828ea6b59def42684a769413aa9f7017779a472977cb5fa04b03c9728c9c28\'}, {\'path\': \'cache_zip_manifest.json\', \'bytes\': 1448259, \'sha256\': \'aa3be67abdd715c74c597e3a02b98cda01d7b59d0ecf50fa465c15ce7433f325\'}, {\'path\': \'port/plan.csv\', \'bytes\': 460297, \'sha256\': \'b17605be3bcc36b00c7974c7084b952b8309bf08b77cc0d887c701cc63f107cd\'}, {\'path\': \'repack_receipt.json\', \'bytes\': 686, \'sha256\': \'0993f73b1c00646bec20e9edf89766098a06e55df56ce27c7854cc55fb15befb\'}, {\'path\': \'source_cache_summary.json\', \'bytes\': 406, \'sha256\': \'5505d293529518a60706f1873e68ce3d6c13ed5f1b25e37c0fdb7c86710f3da0\'}], \'storage_kind\': \'exact_npy_repack\', \'original_npy_byte_identity_claim\': True, \'runtime_seconds\': 1069, \'expected_images\': 10796}]\nHTTP_SOURCE_INDEX_SHA=\'0e450019b7f37a39701d78be46f1ea39b76614f90948c6096f0a1fe3eb2544b7\'\nHTTP_FEATURE_IDENTITY=\'7f0e16dc1c34e8c0920f567db2ebb68f55c470c09ce77dd7c0aa238cd5ec6c77\'\ndef _load_private(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\n    return module\ndef _http_prepare_source(data,manifest,code):\n    import sys\n    sys.path.insert(0,str(code))\n    scientist=_load_private(\'http_v2_verified_science\',code/\'sharded_frozen_extract.py\')\n    plan=scientist.validate_global(data,SETTINGS[\'global_manifest_sha256\'])\n    identity,_=scientist.make_identity(plan,SETTINGS[\'config\'],SETTINGS[\'assets_manifest_sha256\'])\n    if identity!=HTTP_FEATURE_IDENTITY:raise RuntimeError(\'HTTP V2 verified global identity differs\')\n    job=Path(os.environ[\'RSNA_HTTP_JOB_DIR\'])\n    approval=_load_private(\'http_owner_gate\',os.environ[\'RSNA_HTTP_APPROVAL\'])\n    forecast=_load_private(\'http_forecast\',os.environ[\'RSNA_HTTP_FORECAST\'])\n    projection=approval.validate(json.loads((job/\'http_launch_approval.json\').read_text()),HTTP_SPEC[\'shard\'],\n        SETTINGS[\'max_seconds\'],HTTP_FEATURE_IDENTITY,HTTP_SOURCE_INDEX_SHA,forecast.project,HTTP_SOURCE_INDEX)\n    Path(\'/kaggle/working/http_route_forecast.json\').write_text(json.dumps(projection,indent=2))\n    helper=_load_private(\'http_receipt_transport\',os.environ[\'RSNA_HTTP_TRANSPORT\'])\n    downloader=_load_private(\'http_cache_downloader\',os.environ[\'RSNA_HTTP_DOWNLOADER\'])\n    root,receipt=helper.prepare_view(data,Path(os.environ[\'RSNA_HTTP_OWNED_TMP\'])/\'source_view\',HTTP_SPEC,\n        manifest,downloader.download,job/\'cache_read_job.json\',\'/kaggle/working/http_cache_download.json\',\n        BOOTSTRAP_MONOTONIC+SETTINGS[\'max_seconds\'])\n    Path(\'/kaggle/working/http_cache_transport.json\').write_text(json.dumps(receipt,indent=2))\nCOMPRESSED_PAYLOAD=\'...\'\nSETTINGS={\'mode\': \'extract\', \'shards\': [2], \'config\': {\'experiment\': \'orthofoundation_frozen_attention_http_v2\', \'encoder\': \'dinov3_vitl16\', \'extraction_variant\': \'author_no_rope\', \'feature_dim\': 1024, \'checkpoint_sha256\': \'385a775822107b68eaa486336feb982e1ce7bd6d4e8c03ceb482a0bf546f2ff9\', \'checkpoint_bytes\': 1213056638, \'resolution\': 224, \'slices_per_slot\': 4, \'fov\': 0.92, \'image_batch_size\': 16, \'workers\': 2, \'pilot_studies\': 16, \'max_extract_seconds\': 9300, \'head_kind\': \'attention\', \'head_dim\': 256, \'head_dropout\': 0.2, \'head_epochs\': 12, \'head_batch_size\': 64, \'head_lr\': 0.001, \'head_weight_decay\': 0.01, \'weak_holdout_fold\': 0, \'seed\': 42}, \'global_manifest_sha256\': \'b2a87794d768ca3f6ee3968345f840f1028cff004708a2b3e0cfdfcfb2c15e60\', \'assets_manifest_sha256\': \'6feeb03c976483c2b91cca667f4fdbefa998d3620a93eb6006de22a13cbdeb4f\', \'metadata_dataset\': \'alanchoo/rsna-knee-ortho-global-metadata-v1\', \'assets_dataset\': \'alanchoo/orthofoundation-l-assets-v1\', \'cache_kernel_ids\': [\'alanchoo/rsna-knee-cnxt-cache-2-zip\'], \'partial_kernels\': [], \'cache_transport\': \'notebooks\', \'cache_dataset_specs\': [], \'resume_kernel\': None, \'max_seconds\': 1236, \'expected_images_for_shards\': 12580, \'planned_extraction_jobs\': 7, \'total_frozen_budget_seconds\': 9300, \'head_fit_budget_seconds\': 1200, \'head_job_reserve_seconds\': 1500, \'source_sha256\': {\'orthofoundation/__init__.py\': \'f3df43207f5d18497b4f698aa91c746bf81dcd4edb74b81784136acd8aa52cf1\', \'orthofoundation/cache.py\': \'700767576eb7b6d9dfa22f77bdf2088411ba4bdf6036c80db58bf608c96f7485\', \'orthofoundation/data.py\': \'f3b63ab922f0452c065126dc2ef1c87442a0b988dff29d25c201666a8a5310aa\', \'orthofoundation/encoder.py\': \'86defa5f943bfc161603c7b377bbf08a9d5c2a270d01b1df5c039ab59cff7ecb\', \'orthofoundation/heads.py\': \'c814b1bcbf1ac1322b717c1f1e6ba7b9173357e956e9104b2c77d0351a833f1f\', \'orthofoundation/run.py\': \'740c2e2599516b1876e7d7226bb0f11c51205cbdaec0b6eebf44dc740c659e10\', \'orthofoundation/train.py\': \'928121a030aabf4fc58594da967ad98ff43a7ce915474d69a909ab09652a7dc1\', \'sharded_frozen_extract.py\': \'86265241778c40a3822c2a8d11f4f3f746a48b47101def338120dd86937ee4f2\'}, \'read_job_dataset\': \'alanchoo/rsna-knee-ortho-http-shard-2-read-v2\', \'source_index_sha256\': \'0e450019b7f37a39701d78be46f1ea39b76614f90948c6096f0a1fe3eb2544b7\', \'expected_feature_identity\': \'7f0e16dc1c34e8c0920f567db2ebb68f55c470c09ce77dd7c0aa238cd5ec6c77\', \'prepared_only\': True}\ntry:\n    bootstrap_checkpoint(\'imports\')\n    from pathlib import Path\n    import base64,hashlib,json,math,os,re,shutil,subprocess,sys,tarfile,tempfile,time,zlib\n    if SETTINGS.get(\'cache_transport\')==\'datasets\':\n        _RSNA_CACHE_DATASET_SCRATCH=Path(tempfile.mkdtemp(prefix=\'rsna-knee-cache-mount-\',dir=\'/tmp\'))\n    def validate_cache_dataset_manifest(manifest, dataset_id, global_manifest, global_root):\n        """A transport mirror may change packaging, never the global cache receipt."""\n        shard = manifest.get("shard")\n        if type(shard) is not int or shard not in range(7):\n            raise RuntimeError("Dataset mirror shard must be within 0..6")\n        if manifest.get("schema") != "rsna_knee_cache_dataset_mirror_v1":\n            raise RuntimeError("Unknown cache dataset mirror schema")\n        dataset = manifest.get("dataset", {})\n        if dataset.get("identifier") != dataset_id or type(dataset.get("version_number")) is not int or dataset["version_number"] < 1:\n            raise RuntimeError("Dataset mirror identifier/version differs from pin")\n        source = manifest.get("source", {})\n        plan = next(row for row in global_manifest["source_plan"] if row["shard"] == shard)\n        if source.get("kernel") != plan["kernel_id"] or type(source.get("version_number")) is not int or source["version_number"] < 1:\n            raise RuntimeError("Dataset mirror source kernel/version differs from global provenance")\n        pin = next(row for row in global_manifest["cache_input_fingerprints"] if row["plan_sha256"] == plan["plan_sha256"])\n        if manifest.get("fingerprint") != pin:\n            raise RuntimeError("Dataset mirror fingerprint differs from global pin")\n        archive = manifest.get("archive", {})\n        summary = json.loads((Path(global_root) / f"cache_receipts/shard_{shard}/cache_summary.json").read_text())\n        if (archive.get("filename") != "cache384_train.zip.bin" or archive.get("original_name") != "cache384_train.zip"\n                or archive.get("sha256") != pin["archive_sha256"] or archive.get("bytes") != summary["storage_bytes"]):\n            raise RuntimeError("Dataset mirror archive bytes/SHA differ from global pin")\n        receipt_archive = manifest.get("receipt_archive", {})\n        if (receipt_archive.get("filename") != "cache_receipts.tar.gz.bin"\n                or type(receipt_archive.get("bytes")) is not int or receipt_archive["bytes"] <= 0\n                or not re.fullmatch(r"[0-9a-f]{64}", receipt_archive.get("sha256", ""))):\n            raise RuntimeError("Dataset mirror receipt archive is invalid")\n        prefix = f"cache_receipts/shard_{shard}/"\n        expected = {row["path"][len(prefix):]: {"path": row["path"][len(prefix):], "bytes": row["bytes"], "sha256": row["sha256"]}\n                    for row in global_manifest["members"] if row["path"].startswith(prefix)}\n        members = manifest.get("receipt_members", [])\n        if len(members) != len(expected) or {row.get("path"): row for row in members} != expected:\n            raise RuntimeError("Dataset mirror receipt members differ from global metadata")\n        for name in expected:\n            path = Path(name)\n            if path.is_absolute() or ".." in path.parts:\n                raise RuntimeError("Unsafe dataset mirror receipt path")\n        return shard\n\n    def stream_sha256(path, deadline=None):\n        """Bound memory and include transport validation in the Python-stage budget."""\n        digest = hashlib.sha256()\n        with Path(path).open("rb") as stream:\n            while True:\n                if deadline is not None and time.monotonic() >= deadline:\n                    raise RuntimeError("Dataset archive hashing exhausted Python-stage budget")\n                chunk = stream.read(8 * 1024 * 1024)\n                if not chunk:\n                    break\n                digest.update(chunk)\n        return digest.hexdigest()\n\n    def prepare_cache_dataset(source, target, spec, global_manifest, global_root, unpack_archive, checkpoint, deadline):\n        """Expose exact bytes under the unchanged loader\'s canonical ZIP filename."""\n        started = time.monotonic()\n        manifest_file = Path(source) / "cache_dataset_manifest.json"\n        if stream_sha256(manifest_file, deadline) != spec["manifest_sha256"]:\n            raise RuntimeError("Cache dataset version manifest SHA differs from prepared pin")\n        manifest = json.loads(manifest_file.read_text())\n        shard = validate_cache_dataset_manifest(manifest, spec["dataset_id"], global_manifest, global_root)\n        if manifest != spec["manifest"] or shard != spec["shard"]:\n            raise RuntimeError("Cache dataset version manifest differs from prepared source/version")\n        archive = Path(source) / manifest["archive"]["filename"]\n        if not archive.is_file() or archive.stat().st_size != manifest["archive"]["bytes"]:\n            raise RuntimeError("Mounted dataset archive size differs from pin")\n        checkpoint("dataset_cache_full_archive_hash")\n        hash_started = time.monotonic()\n        if stream_sha256(archive, deadline) != manifest["archive"]["sha256"]:\n            raise RuntimeError("Mounted dataset archive SHA differs from pin")\n        hash_seconds = time.monotonic() - hash_started\n        checkpoint("dataset_cache_receipts_unpack")\n        if Path(target).exists():\n            raise RuntimeError("Dataset cache scratch root already exists; refuse stale files")\n        Path(target).parent.mkdir(parents=True, exist_ok=True)\n        staging = Path(target).parent / (Path(target).name + "_receipts_verified")\n        if staging.exists():\n            raise RuntimeError("Dataset receipt verification staging already exists")\n        unpack_archive(Path(source) / manifest["receipt_archive"]["filename"], staging, manifest["receipt_archive"],\n                       [row["path"] for row in manifest["receipt_members"]])\n        for row in manifest["receipt_members"]:\n            path = staging / row["path"]\n            if path.stat().st_size != row["bytes"] or stream_sha256(path, deadline) != row["sha256"]:\n                raise RuntimeError("Dataset receipt bytes/SHA differ from global pin before copy")\n        Path(target).mkdir()\n        for row in manifest["receipt_members"]:\n            destination = Path(target) / row["path"]\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(staging / row["path"], destination)\n            if stream_sha256(destination, deadline) != row["sha256"]:\n                raise RuntimeError("Copied dataset receipt bytes/SHA differ from global pin")\n        shutil.rmtree(staging)\n        (Path(target) / manifest["archive"]["original_name"]).symlink_to(archive.resolve())\n        receipt = {"shard": shard, "dataset_id": spec["dataset_id"], "dataset_version_number": manifest["dataset"]["version_number"],\n                   "manifest_sha256": spec["manifest_sha256"], "source": manifest["source"],\n                   "archive_sha256": manifest["archive"]["sha256"], "archive_bytes": manifest["archive"]["bytes"],\n                   "archive_hash_seconds": hash_seconds, "transport_validation_seconds": time.monotonic() - started,\n                   "timer_boundary": "Included in inherited Python bootstrap timer; pre-Python Kaggle mounting excluded"}\n        return Path(target), receipt\n\n    CODE=Path(\'/kaggle/working/sharded_source\'); CODE.mkdir(exist_ok=True)\n    bootstrap_checkpoint(\'vendoring\')\n    for name,encoded in json.loads(zlib.decompress(base64.b64decode(COMPRESSED_PAYLOAD))).items():\n        destination=CODE/name; destination.parent.mkdir(exist_ok=True); destination.write_bytes(base64.b64decode(encoded))\n        if hashlib.sha256(destination.read_bytes()).hexdigest()!=SETTINGS[\'source_sha256\'][name]:\n            raise RuntimeError(\'Vendored sharded source SHA mismatch: \'+name)\n    (CODE/\'config.json\').write_text(json.dumps(SETTINGS[\'config\'],indent=2))\n    INPUT=Path(\'/kaggle/input\')\n    def find_input(identifier,marker,kind=\'datasets\'):\n        if kind==\'notebooks\' and marker==\'cache_summary.json\' and identifier==HTTP_SPEC[\'cache_kernel_id\']:\n            return Path(os.environ[\'RSNA_HTTP_OWNED_TMP\'])/\'source_view\'\n        owner,slug=identifier.split(\'/\')\n        candidates=[INPUT/kind/owner/slug,INPUT/slug,INPUT/\'kernels\'/owner/slug,INPUT/owner/slug]\n        found=[path for path in candidates if (path/marker).is_file()]\n        if not found:\n            for pattern in [f\'*/{slug}\',f\'*/*/{slug}\',f\'*/*/*/{slug}\']:\n                found.extend(path for path in INPUT.glob(pattern) if (path/marker).is_file())\n        found=list(dict.fromkeys(found))\n        if len(found)!=1:raise RuntimeError(f\'Missing/ambiguous private source {identifier}: {found}\')\n        return found[0]\n    def unpack(archive,target,expected,allowed_members=None):\n        if archive.stat().st_size!=expected[\'bytes\'] or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected[\'sha256\']:\n            raise RuntimeError(\'Private archive size/SHA differs from pin\')\n        target.mkdir(exist_ok=True)\n        with tarfile.open(archive,\'r:gz\') as tar:\n            members=tar.getmembers(); files=[]\n            for member in members:\n                path=Path(member.name)\n                if path.is_absolute() or \'..\' in path.parts or member.issym() or member.islnk():\n                    raise RuntimeError(\'Unsafe private archive member\')\n                if member.isfile():files.append(member.name)\n            if len(files)!=len(set(files)) or (allowed_members is not None and set(files)!=set(allowed_members)):\n                raise RuntimeError(\'Private archive members differ from manifest\')\n            tar.extractall(target,filter=\'data\')\n    bootstrap_checkpoint(\'global_metadata_resolution\')\n    metadata=find_input(SETTINGS[\'metadata_dataset\'],\'global_metadata_manifest.json\')\n    manifest_file=metadata/\'global_metadata_manifest.json\'\n    if hashlib.sha256(manifest_file.read_bytes()).hexdigest()!=SETTINGS[\'global_manifest_sha256\']:\n        raise RuntimeError(\'Global private manifest differs from prepared pin\')\n    manifest=json.loads(manifest_file.read_text())\n    DATA=Path(\'/kaggle/working/global_metadata\')\n    bootstrap_checkpoint(\'global_metadata_unpack\')\n    unpack(metadata/manifest[\'archive\'][\'filename\'],DATA,manifest[\'archive\'],[row[\'path\'] for row in manifest[\'members\']])\n    shutil.copyfile(manifest_file,DATA/\'global_metadata_manifest.json\')\n    bootstrap_checkpoint(\'verified_global_HTTP_view\')\n    _http_prepare_source(DATA,manifest,CODE)\n    cmd=[sys.executable,str(CODE/\'sharded_frozen_extract.py\'),\'--mode\',SETTINGS[\'mode\'],\n         \'--global-root\',str(DATA),\'--global-manifest-sha256\',SETTINGS[\'global_manifest_sha256\'],\n         \'--assets-manifest-sha256\',SETTINGS[\'assets_manifest_sha256\'],\'--config-json\',str(CODE/\'config.json\'),\n         \'--output-dir\',\'/kaggle/working/run\',\'--max-seconds\',str(SETTINGS[\'max_seconds\']),\n         \'--planned-extraction-jobs\',str(SETTINGS[\'planned_extraction_jobs\'])]\n    if SETTINGS[\'mode\'] in (\'pilot\',\'extract\'):\n        bootstrap_checkpoint(\'encoder_assets_resolution\')\n        assets=find_input(SETTINGS[\'assets_dataset\'],\'asset_manifest.json\')\n        asset_file=assets/\'asset_manifest.json\'\n        if hashlib.sha256(asset_file.read_bytes()).hexdigest()!=SETTINGS[\'assets_manifest_sha256\']:\n            raise RuntimeError(\'Encoder asset manifest differs from prepared pin\')\n        asset_manifest=json.loads(asset_file.read_text()); dinov3=asset_manifest[\'dinov3_source\']\n        bootstrap_checkpoint(\'encoder_source_unpack\')\n        source_archive=assets/dinov3[\'filename\']\n        unpack(source_archive,CODE,{\'bytes\':source_archive.stat().st_size,\'sha256\':dinov3[\'sha256\']})\n        bootstrap_checkpoint(\'owned_cache_resolution\')\n        if SETTINGS.get(\'cache_transport\',\'notebooks\')==\'datasets\':\n            roots=[]; transport=[]\n            for spec in SETTINGS[\'cache_dataset_specs\']:\n                source=find_input(spec[\'dataset_id\'],\'cache_dataset_manifest.json\')\n                root,receipt=prepare_cache_dataset(source,_RSNA_CACHE_DATASET_SCRATCH/f"shard_{spec[\'shard\']}",\n                    spec,manifest,DATA,unpack,bootstrap_checkpoint,BOOTSTRAP_MONOTONIC+SETTINGS[\'max_seconds\'])\n                roots.append(root); transport.append(receipt)\n                Path(\'/kaggle/working/cache_dataset_transport.json\').write_text(json.dumps({\'route\':\'datasets\',\'mirrors\':transport},indent=2))\n                print(json.dumps({\'event\':\'cache_dataset_transport_verified\',**receipt}),flush=True)\n        else:\n            roots=[find_input(identifier,\'cache_summary.json\',\'notebooks\') for identifier in SETTINGS[\'cache_kernel_ids\']]\n        cmd+=[\'--shards\',*[str(shard) for shard in SETTINGS[\'shards\']],\'--cache-roots\',*[str(root) for root in roots],\n              \'--dinov3-repo\',str(CODE/dinov3[\'archive_root\']),\'--checkpoint\',str(assets/asset_manifest[\'checkpoint\'][\'filename\']),\n              \'--assets-manifest\',str(asset_file)]\n        if SETTINGS[\'resume_kernel\']:\n            prior=find_input(SETTINGS[\'resume_kernel\'],\'run/partial_manifest.json\',\'notebooks\')\n            cmd+=[\'--resume-partial\',str(prior/\'run\')]\n    else:\n        bootstrap_checkpoint(\'partial_features_resolution\')\n        roots=[find_input(identifier,\'run/partial_manifest.json\',\'notebooks\') for identifier in SETTINGS[\'partial_kernels\']]\n        cmd+=[\'--partial-roots\',*[str(root/\'run\') for root in roots]]\n    bootstrap_checkpoint(\'gpu_selection\')\n    import torch\n    if not torch.cuda.is_available():raise RuntimeError(\'No CUDA GPU; refuse CPU fallback\')\n    free=[torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]\n    env=dict(os.environ); env[\'PYTHONPATH\']=str(CODE)+os.pathsep+env.get(\'PYTHONPATH\',\'\')\n    env[\'OMP_NUM_THREADS\']=\'1\'; env[\'OPENBLAS_NUM_THREADS\']=\'1\'\n    env[\'CUDA_VISIBLE_DEVICES\']=str(max(range(len(free)),key=lambda index:free[index]))\n    env[\'RSNA_SHARDED_BOOTSTRAP_START\']=str(BOOTSTRAP_MONOTONIC)\n    print(json.dumps({\'mode\':SETTINGS[\'mode\'],\'shards\':SETTINGS[\'shards\'],\'max_seconds\':SETTINGS[\'max_seconds\'],\n                      \'launch\':cmd,\'gpu_free_gb\':[value/1e9 for value in free]}),flush=True)\n    bootstrap_checkpoint(\'launch\')\n    remaining=SETTINGS[\'max_seconds\']-(time.monotonic()-BOOTSTRAP_MONOTONIC)\n    if remaining<=0:raise RuntimeError(\'Private launcher Python-stage budget exhausted before child launch\')\n    subprocess.run(cmd,env=env,check=True,timeout=remaining)\n    bootstrap_checkpoint(\'report\')\n    name={\'pilot\':\'pilot_report.json\',\'extract\':\'extraction_report.json\',\'merge-train\':\'training_report.json\'}[SETTINGS[\'mode\']]\n    report=json.loads((Path(\'/kaggle/working/run\')/name).read_text())\n    if SETTINGS[\'mode\']==\'extract\' and not report[\'extraction_complete_for_shards\']:raise RuntimeError(\'Shard feature extraction incomplete\')\n    if SETTINGS[\'mode\']==\'merge-train\' and not report[\'head_training_complete\']:raise RuntimeError(\'Merged head training incomplete\')\n    print(json.dumps(report,indent=2),flush=True)\n    bootstrap_checkpoint(\'complete\')\nexcept BaseException as bootstrap_error:\n    bootstrap_save_failure(bootstrap_error)\n    raise\n'
TRANSPORT_SOURCES={'download.py': 'import hashlib,json,re,time,zlib\nfrom pathlib import Path\nimport requests\nfrom transfer_reviewed import TransferStopped,RejectAllCookies,validate_job,atomic_report,DOWNLOAD_RANGE_BYTES\n\ndef download(job_path,archive,output,deadline,expected,expected_origin,shard,runtime,allow_local_fixture=False):\n    start=time.monotonic();report={\'complete\':False,\'route\':\'direct_http\',\'origin\':expected_origin,\n        \'urls_or_credentials_written_to_output\':False};session=None;stage=\'validate\'\n    archive=Path(archive)\n    def check_time():\n        if time.monotonic()>=deadline:raise TransferStopped(\'walltime_budget\')\n    try:\n        job=json.loads(Path(job_path).read_text())\n        if set(job)!={\'job_id\',\'filename\',\'expected_bytes\',\'expected_sha256\',\'read_url\',\'max_seconds\',\'source_origin\'}:\n            raise TransferStopped(\'invalid_read_only_job_fields\')\n        if job[\'source_origin\']!=expected_origin or job[\'job_id\']!=f\'shard_{shard}\':raise TransferStopped(\'source_origin_mismatch\')\n        if job[\'expected_bytes\']!=expected[\'bytes\'] or job[\'expected_sha256\']!=expected[\'sha256\']:\n            raise TransferStopped(\'immutable_cache_pin_mismatch\')\n        if job[\'max_seconds\']!=runtime:raise TransferStopped(\'runtime_pin_mismatch\')\n        validation={k:v for k,v in job.items() if k!=\'source_origin\'}\n        # Constant validation-only placeholder reuses the reviewed URL/field guards.\n        # This module has no upload branch and never contacts this placeholder.\n        validation[\'put_url\']=\'https://storage.googleapis.com/disabled-validation-only\'\n        validate_job(validation,allow_local_fixture)\n        total=job[\'expected_bytes\'];report.update(expected_bytes=total,expected_sha256=job[\'expected_sha256\'])\n        archive.parent.mkdir(parents=True,exist_ok=True)\n        session=requests.Session();session.trust_env=False;session.headers.clear();session.auth=None\n        session.cookies.set_policy(RejectAllCookies())\n        stage = "download"\n        digest, downloaded = hashlib.sha256(), 0\n        report.update(download_range_checks=[], download_retry_count=0, download_range_max_bytes=DOWNLOAD_RANGE_BYTES)\n        with archive.open("w+b") as stream:\n            for offset in range(0, total, DOWNLOAD_RANGE_BYTES):\n                end = min(total, offset + DOWNLOAD_RANGE_BYTES) - 1\n                expected_part = end - offset + 1\n                for attempt in range(3):\n                    check_time()\n                    stream.seek(offset)\n                    stream.truncate(offset)\n                    part_digest, received, crc = digest.copy(), 0, 0\n                    try:\n                        with session.get(job["read_url"], stream=True, timeout=(15, 60), allow_redirects=False,\n                                         headers={"Accept-Encoding": "identity", "Range": f"bytes={offset}-{end}"}) as response:\n                            report["download_http_status"] = response.status_code\n                            if response.status_code != 206:\n                                raise TransferStopped("download_range_http_failure")\n                            if response.headers.get("Content-Encoding", "identity") != "identity":\n                                raise TransferStopped("unexpected_transfer_encoding")\n                            cr = re.fullmatch(r"bytes (\\d+)-(\\d+)/(\\d+)", response.headers.get("Content-Range", ""))\n                            if not cr or tuple(map(int, cr.groups())) != (offset, end, total):\n                                raise TransferStopped("download_content_range_mismatch")\n                            length = response.headers.get("Content-Length")\n                            if length is not None and int(length) != expected_part:\n                                raise TransferStopped("download_header_size_mismatch")\n                            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):\n                                check_time()\n                                if not chunk:\n                                    continue\n                                received += len(chunk)\n                                if received > expected_part:\n                                    raise TransferStopped("download_range_exceeded_size")\n                                part_digest.update(chunk)\n                                crc = zlib.crc32(chunk, crc)\n                                stream.write(chunk)\n                        if received != expected_part:\n                            raise TransferStopped("download_range_body_size_mismatch")\n                        digest = part_digest\n                        downloaded += received\n                        report["download_range_checks"].append({"start": offset, "end": end, "bytes": received,\n                                                                "crc32_diagnostic": f"{crc:08x}"})\n                        break\n                    except requests.RequestException:\n                        report["download_retry_count"] += 1\n                        if attempt == 2:\n                            raise TransferStopped("download_range_transport_attempt_budget") from None\n        report.update(downloaded_bytes=downloaded, downloaded_sha256=digest.hexdigest())\n        if downloaded != total or digest.hexdigest() != job["expected_sha256"]:\n            raise TransferStopped("download_size_or_sha_mismatch")\n        report["download_sha256_verified"] = True\n        check_time()\n        report[\'complete\']=True\n    except BaseException as error:\n        report.update(complete=False,failure_stage=stage,failure_code=error.code if isinstance(error,TransferStopped)\n                      else \'redacted_download_exception\',exception_type=type(error).__name__)\n    finally:\n        if session is not None:session.close()\n        if not report[\'complete\']:archive.unlink(missing_ok=True)\n        report[\'seconds\']=time.monotonic()-start;atomic_report(report,output)\n    if not report[\'complete\']:raise TransferStopped(\'http_cache_download_failed\')\n    return report\n', 'transport.py': '"""Receipt views for exact HTTP ZIP bytes; no API, signing or account auth."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport time\n\nCAPS = [1204, 1231, 1236, 1213, 1216, 1228, 1069]\nIMAGES = [12240, 12528, 12580, 12332, 12364, 12496, 10796]\n\n\ndef file_sha(path, deadline=None):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        while True:\n            if deadline is not None and time.monotonic() >= deadline:\n                raise RuntimeError(\'Receipt verification exhausted stage deadline\')\n            block = stream.read(8 * 1024 * 1024)\n            if not block:\n                break\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef expected_spec(manifest, root, shard, runtime, images):\n    if type(shard) is not int or shard not in range(7):\n        raise RuntimeError(\'Invalid HTTP shard\')\n    if runtime != CAPS[shard] or images != IMAGES[shard]:\n        raise RuntimeError(\'HTTP shard cap or global selected-image count differs\')\n    source = next(row for row in manifest[\'source_plan\'] if row[\'shard\'] == shard)\n    pin = next(row for row in manifest[\'cache_input_fingerprints\'] if row[\'plan_sha256\'] == source[\'plan_sha256\'])\n    summary = json.loads((Path(root) / f\'cache_receipts/shard_{shard}/cache_summary.json\').read_text())\n    prefix = f\'cache_receipts/shard_{shard}/\'\n    members = [{**row, \'path\': row[\'path\'][len(prefix):]} for row in manifest[\'members\'] if row[\'path\'].startswith(prefix)]\n    if not members or len({row[\'path\'] for row in members}) != len(members):\n        raise RuntimeError(\'Missing or duplicate globally pinned shard receipts\')\n    origin = ({\'kind\': \'dataset\', \'identifier\': \'alanchoo/rsna-knee-cache-0-bin-v1\',\n               \'version_number\': 1, \'filename\': \'cache384_train.zip.bin\'} if shard == 0 else\n              {\'kind\': \'kernel\', \'identifier\': source[\'kernel_id\'],\n               \'version_number\': 1, \'filename\': \'cache384_train.zip\'})\n    return {\'shard\': shard, \'cache_kernel_id\': source[\'kernel_id\'], \'origin\': origin,\n            \'archive\': {\'filename\': \'cache384_train.zip\', \'bytes\': summary[\'storage_bytes\'],\n                        \'sha256\': pin[\'archive_sha256\']},\n            \'job_filename_alias\': \'cache384_train.zip.bin\', \'job_id\': f\'shard_{shard}\',\n            \'fingerprint\': pin, \'receipt_members\': members, \'storage_kind\': source[\'storage_kind\'],\n            \'original_npy_byte_identity_claim\': False if shard == 5 else True,\n            \'runtime_seconds\': runtime, \'expected_images\': images}\n\n\ndef validate_spec(spec, manifest, root):\n    expected = expected_spec(manifest, root, spec[\'shard\'], spec[\'runtime_seconds\'], spec[\'expected_images\'])\n    if spec != expected:\n        raise RuntimeError(\'HTTP cache spec differs from globally pinned original ZIP source\')\n    if not 0 < spec[\'archive\'][\'bytes\'] <= 11_000_000_000:\n        raise RuntimeError(\'HTTP archive size outside reviewed downloader bounds\')\n    for member in spec[\'receipt_members\']:\n        path = Path(member[\'path\'])\n        if path.is_absolute() or \'..\' in path.parts or path == Path(\'.\'):\n            raise RuntimeError(\'Unsafe receipt member path\')\n\n\ndef prepare_view(root, target, spec, manifest, download, job_path, report_path, deadline,\n                 allow_local_fixture=False):\n    """Small receipts copy exactly; downloaded ZIP is already in owned /tmp.\n\n    The downloader\'s streaming full SHA is authoritative. The unchanged core\n    loader subsequently verifies metadata/receipt fingerprint and ZIP index/CRC.\n    No second 10 GB hash, archive rewrite, compression or large working symlink.\n    """\n    started = time.monotonic()\n    validate_spec(spec, manifest, root)\n    target = Path(target)\n    if target.exists():\n        raise RuntimeError(\'HTTP receipt view already exists\')\n    source = Path(root) / f"cache_receipts/shard_{spec[\'shard\']}"\n    # Check all source receipts before copying any member or starting HTTP.\n    for member in spec[\'receipt_members\']:\n        path = source / member[\'path\']\n        if (path.is_symlink() or not path.is_file() or path.stat().st_size != member[\'bytes\']\n                or file_sha(path, deadline) != member[\'sha256\']):\n            raise RuntimeError(\'Global shard receipt bytes/SHA changed before view preparation\')\n    target.mkdir(parents=True)\n    for member in spec[\'receipt_members\']:\n        destination = target / member[\'path\']\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(source / member[\'path\'], destination)\n        if destination.stat().st_size != member[\'bytes\'] or file_sha(destination, deadline) != member[\'sha256\']:\n            raise RuntimeError(\'Copied HTTP receipt bytes/SHA changed\')\n    receipt_seconds = time.monotonic() - started\n    result = download(job_path, target / spec[\'archive\'][\'filename\'], report_path, deadline,\n                      spec[\'archive\'], spec[\'origin\'], spec[\'shard\'], spec[\'runtime_seconds\'],\n                      allow_local_fixture)\n    if not result.get(\'complete\') or not result.get(\'download_sha256_verified\'):\n        raise RuntimeError(\'HTTP ZIP was not fully downloaded and SHA verified\')\n    return target, {\'shard\': spec[\'shard\'], \'origin\': spec[\'origin\'], \'bytes\': spec[\'archive\'][\'bytes\'],\n                    \'sha256\': spec[\'archive\'][\'sha256\'], \'receipt_seconds\': receipt_seconds,\n                    \'download_seconds\': result[\'seconds\'], \'full_streaming_SHA_verified\': True,\n                    \'second_full_archive_hash\': False, \'storage_kind\': spec[\'storage_kind\'],\n                    \'original_npy_byte_identity_claim\': spec[\'original_npy_byte_identity_claim\']}\n', 'forecast.py': '"""Extrinsic whole-route forecast; no authority to start GPU work."""\nimport math\n\nEXPECTED_IMAGES = [12240, 12528, 12580, 12332, 12364, 12496, 10796]\nCAPS = [1204, 1231, 1236, 1213, 1216, 1228, 1069]\nTOTAL_IMAGES = 85336\nPILOT_CAP = 900\nTOTAL_CAP = 9300\n\n\ndef project(specs, http_bytes_per_second, non_transport_startup_seconds, gpu_images_per_second):\n    numbers = [http_bytes_per_second, non_transport_startup_seconds, gpu_images_per_second]\n    if (not all(type(value) in (int, float) and math.isfinite(value) for value in numbers)\n            or http_bytes_per_second <= 0 or gpu_images_per_second <= 0 or non_transport_startup_seconds < 0):\n        raise RuntimeError(\'Invalid measured HTTP/GPU throughput or non-transport startup\')\n    if len(specs) != 7 or [row[\'shard\'] for row in specs] != list(range(7)):\n        raise RuntimeError(\'Forecast requires all seven ordered immutable archive pins\')\n    if [row[\'expected_images\'] for row in specs] != EXPECTED_IMAGES:\n        raise RuntimeError(\'Forecast global selected-image counts differ\')\n    if [row[\'runtime_seconds\'] for row in specs] != CAPS:\n        raise RuntimeError(\'Forecast original shard caps differ\')\n    archive_bytes = [row[\'archive\'][\'bytes\'] for row in specs]\n    if any(type(value) is not int or value <= 0 for value in archive_bytes):\n        raise RuntimeError(\'Invalid forecast archive byte counts\')\n    jobs = []\n    for row in specs:\n        http = row[\'archive\'][\'bytes\'] / http_bytes_per_second\n        gpu = row[\'expected_images\'] / gpu_images_per_second\n        total = http + non_transport_startup_seconds + gpu\n        jobs.append({\'shard\': row[\'shard\'], \'http_seconds\': http, \'gpu_seconds\': gpu,\n                     \'non_transport_startup_seconds\': non_transport_startup_seconds,\n                     \'projected_seconds\': total, \'cap_seconds\': row[\'runtime_seconds\'],\n                     \'fits_cap\': total <= row[\'runtime_seconds\']})\n    transport = sum(archive_bytes) / http_bytes_per_second\n    startup = non_transport_startup_seconds * 7\n    gpu = TOTAL_IMAGES / gpu_images_per_second\n    total = PILOT_CAP + transport + startup + gpu\n    return {\'schema\': \'http_sharded_v2_forecast\', \'all_archive_bytes\': sum(archive_bytes),\n            \'http_bytes_per_second\': http_bytes_per_second, \'gpu_images_per_second\': gpu_images_per_second,\n            \'non_transport_startup_seconds_per_job\': non_transport_startup_seconds,\n            \'transport_seconds_all_seven\': transport, \'startup_seconds_all_seven\': startup,\n            \'GPU_feature_seconds_all_85336_images\': gpu, \'pilot_reserved_seconds\': PILOT_CAP,\n            \'projected_total_extraction_seconds\': total, \'total_extraction_cap_seconds\': TOTAL_CAP,\n            \'fits_total\': total <= TOTAL_CAP, \'fits_every_shard\': all(row[\'fits_cap\'] for row in jobs),\n            \'head_fit_seconds\': 1200, \'head_job_reserve_seconds\': 1500,\n            \'projected_total_plus_head_job_seconds\': total + 1500, \'shards\': jobs,\n            \'launch_authorized\': False,\n            \'remaining_gates\': [\'Actual MRI pilot PASS and sealed measured receipts\', \'Fresh quota before every job\'],\n            \'timer_boundary\': \'Python HTTP, hashing, setup, encoder and export; pre-Python Kaggle mounting excluded\'}\n\n\ndef from_pilot(specs, pilot, download, cache_transport, expected_identity):\n    if (pilot.get(\'feature_identity\') != expected_identity or pilot.get(\'shards\') != [0]\n            or not pilot.get(\'pilot_only\') or pilot.get(\'max_seconds\') != 900\n            or pilot.get(\'head_smoke\', {}).get(\'optimizer_updates\', 0) < 1):\n        raise RuntimeError(\'No matching actual V2 MRI pilot/head-update receipt\')\n    features = pilot[\'feature_extraction\']\n    if not features.get(\'complete\') or features.get(\'images_encoded\', 0) <= 0 or features.get(\'seconds\', 0) <= 0:\n        raise RuntimeError(\'No actual finite completed MRI extraction throughput\')\n    if (not download.get(\'complete\') or not download.get(\'download_sha256_verified\')\n            or download.get(\'expected_bytes\') != specs[0][\'archive\'][\'bytes\']\n            or download.get(\'expected_sha256\') != specs[0][\'archive\'][\'sha256\']\n            or download.get(\'origin\') != specs[0][\'origin\'] or download.get(\'seconds\', 0) <= 0):\n        raise RuntimeError(\'HTTP speed does not belong to exact completed source0 download\')\n    mirrors = cache_transport.get(\'mirrors\', [])\n    if (len(mirrors) != 1 or mirrors[0].get(\'shard\') != 0 or mirrors[0].get(\'archive_hash_seconds\', -1) < 0\n            or mirrors[0].get(\'archive_sha256\') != specs[0][\'archive\'][\'sha256\']\n            or mirrors[0].get(\'archive_bytes\') != specs[0][\'archive\'][\'bytes\']):\n        raise RuntimeError(\'Missing V2 pilot second-hash timing receipt\')\n    # Full route streams SHA once. Keep the pilot\'s measured second hash inside\n    # non-transport startup conservatively; do not silently assume a speedup.\n    startup = pilot[\'budget_projection\'][\'startup_seconds\'] - download[\'seconds\']\n    if not math.isfinite(startup) or startup < 0:\n        raise RuntimeError(\'Pilot transport/startup timing boundaries are inconsistent\')\n    report = project(specs, download[\'expected_bytes\'] / download[\'seconds\'], startup,\n                     features[\'images_encoded\'] / features[\'seconds\'])\n    report[\'pilot_feature_identity\'] = expected_identity\n    report[\'actual_matching_pilot_receipts_verified\'] = True\n    report[\'pilot_second_archive_hash_seconds_retained_in_startup\'] = mirrors[0][\'archive_hash_seconds\']\n    return report\n', 'approval.py': '"""Explicit owner-issued compute gate; source pins come from metadata, not job."""\nimport math\nimport re\nimport time\n\n\ndef validate(approval, shard, runtime, expected_identity, source_index_sha, forecast_function, specs, now=None):\n    if (approval.get(\'schema\') != \'http_sharded_v2_owner_gate\' or approval.get(\'shard\') != shard\n            or approval.get(\'approved\') is not True or approval.get(\'actual_mri_pilot_pass\') is not True\n            or approval.get(\'feature_identity\') != expected_identity\n            or approval.get(\'source_index_sha256\') != source_index_sha\n            or approval.get(\'pilot_kernel\') != \'alanchoo/rsna-knee-ortho-http-cache0-pilot-v2\'):\n        raise RuntimeError(\'Missing matching owner gate after actual V2 MRI pilot PASS\')\n    hashes = approval.get(\'pilot_receipt_sha256\', {})\n    if set(hashes) != {\'pilot_report\', \'http_cache_download\', \'cache_dataset_transport\'} or any(\n            not re.fullmatch(r\'[0-9a-f]{64}\', str(value)) for value in hashes.values()):\n        raise RuntimeError(\'Actual sealed pilot receipt hashes are required\')\n    measurements = approval.get(\'measured\', {})\n    projection = forecast_function(specs, measurements.get(\'http_bytes_per_second\'),\n                                   measurements.get(\'non_transport_startup_seconds\'),\n                                   measurements.get(\'gpu_images_per_second\'))\n    if not projection[\'fits_total\'] or not projection[\'fits_every_shard\']:\n        raise RuntimeError(\'Whole HTTP route forecast exceeds fixed total or shard caps\')\n    quota = approval.get(\'fresh_quota\', {})\n    values = [quota.get(\'observed_unix_seconds\'), quota.get(\'available_seconds\'), quota.get(\'required_seconds\')]\n    if not all(type(value) in (int, float) and math.isfinite(value) for value in values):\n        raise RuntimeError(\'Fresh quota receipt must contain finite measured times and seconds\')\n    observed, available, required = values\n    current = time.time() if now is None else now\n    if not 0 <= current - observed <= 900 or required < runtime + 1500 or available < required:\n        raise RuntimeError(\'Quota receipt is stale or below owner reserve including head job\')\n    return projection\n', 'transfer_reviewed.py': '"""Single-file cloud transfer using temporary URLs; no Kaggle account credential.\n\nPreparation only. The parent creates/retains the blob upload token on the Mac.\nThis utility only receives the exact file\'s read URL and resumable PUT URL.\n"""\nimport argparse\nimport hashlib\nimport json\nimport re\nimport shutil\nimport tempfile\nimport time\nimport zlib\nfrom http.cookiejar import DefaultCookiePolicy\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nimport requests\n\nDOWNLOAD_RANGE_BYTES = 512 * 1024 * 1024\n\n\nclass TransferStopped(Exception):\n    def __init__(self, code):\n        self.code = code\n        super().__init__(code)\n\n\nclass RejectAllCookies(DefaultCookiePolicy):\n    def set_ok(self, cookie, request):\n        return False\n\n    def return_ok(self, cookie, request):\n        return False\n\n\nclass DeadlineReader:\n    """Seekable upload body with a deadline checked on each transport read."""\n    mode = "rb"\n\n    def __init__(self, stream, check_time):\n        self.stream, self.check_time = stream, check_time\n\n    def read(self, size=-1):\n        self.check_time()\n        return self.stream.read(size)\n\n    def tell(self):\n        return self.stream.tell()\n\n    def seek(self, *args):\n        return self.stream.seek(*args)\n\n    def fileno(self):\n        return self.stream.fileno()\n\n\ndef validate_job(job, allow_local_fixture=False):\n    required = {"job_id", "filename", "expected_bytes", "expected_sha256", "read_url", "put_url"}\n    if set(job) - required - {"max_seconds"} or required - set(job):\n        raise TransferStopped("invalid_job_fields")\n    if not re.fullmatch(r"shard_[0-6]", str(job["job_id"])):\n        raise TransferStopped("invalid_job_id")\n    if job["filename"] != "cache384_train.zip.bin":\n        raise TransferStopped("invalid_generic_filename")\n    if type(job["expected_bytes"]) is not int or not 0 < job["expected_bytes"] <= 11_000_000_000:\n        raise TransferStopped("invalid_file_size")\n    if not re.fullmatch(r"[0-9a-f]{64}", str(job["expected_sha256"])):\n        raise TransferStopped("invalid_sha256")\n    if type(job.get("max_seconds", 1800)) is not int or not 1 <= job.get("max_seconds", 1800) <= 7200:\n        raise TransferStopped("invalid_time_budget")\n    for field in ("read_url", "put_url"):\n        parsed = urlsplit(job[field])\n        fixture = allow_local_fixture and parsed.scheme == "http" and parsed.hostname in {"127.0.0.1", "localhost"}\n        allowed_hosts = {"storage.googleapis.com", "www.kaggleusercontent.com"} if field == "read_url" else {"storage.googleapis.com"}\n        if not fixture and (parsed.scheme != "https" or parsed.hostname not in allowed_hosts):\n            raise TransferStopped("unsupported_scoped_storage_host")\n        if parsed.username or parsed.password or parsed.fragment:\n            raise TransferStopped("unsafe_url_structure")\n    if job["read_url"] == job["put_url"]:\n        raise TransferStopped("identical_source_destination")\n\n\ndef received_offset(range_header, total):\n    if range_header is None:\n        return 0\n    match = re.fullmatch(r"bytes=0-(\\d+)", range_header)\n    if not match or not 0 <= int(match[1]) < total:\n        raise TransferStopped("invalid_resumable_range")\n    return int(match[1]) + 1\n\n\ndef atomic_report(report, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".tmp")\n    temporary.write_text(json.dumps(report, indent=2))\n    temporary.replace(destination)\n\n\ndef transfer(job, output, scratch_parent, allow_local_fixture=False):\n    # Secret values never enter the report, progress, filenames, or exceptions.\n    report = {"complete": False, "cloud_account_token_used": False,\n              "urls_or_upload_tokens_written_to_output": False}\n    scratch = None\n    stage = "validate"\n    start = time.monotonic()\n    session = requests.Session()\n    session.trust_env = False\n    session.headers.clear()\n    session.auth = None\n    session.cookies.set_policy(RejectAllCookies())\n    deadline = None\n\n    def check_time():\n        if deadline is not None and time.monotonic() >= deadline:\n            raise TransferStopped("walltime_budget")\n\n    try:\n        validate_job(job, allow_local_fixture)\n        report.update(job_id=job["job_id"], filename=job["filename"],\n                      expected_bytes=job["expected_bytes"], expected_sha256=job["expected_sha256"])\n        total = job["expected_bytes"]\n        deadline = start + job.get("max_seconds", 1800)\n        parent = Path(scratch_parent)\n        parent.mkdir(parents=True, exist_ok=True)\n        if shutil.disk_usage(parent).free < total + 512 * 1024 * 1024:\n            raise TransferStopped("insufficient_scratch_space")\n        scratch = Path(tempfile.mkdtemp(prefix="cache_transfer_", dir=parent))\n        archive = scratch / job["filename"]\n        stage = "download"\n        digest, downloaded = hashlib.sha256(), 0\n        report.update(download_range_checks=[], download_retry_count=0, download_range_max_bytes=DOWNLOAD_RANGE_BYTES)\n        with archive.open("w+b") as stream:\n            for offset in range(0, total, DOWNLOAD_RANGE_BYTES):\n                end = min(total, offset + DOWNLOAD_RANGE_BYTES) - 1\n                expected_part = end - offset + 1\n                for attempt in range(3):\n                    check_time()\n                    stream.seek(offset)\n                    stream.truncate(offset)\n                    part_digest, received, crc = digest.copy(), 0, 0\n                    try:\n                        with session.get(job["read_url"], stream=True, timeout=(15, 60), allow_redirects=False,\n                                         headers={"Accept-Encoding": "identity", "Range": f"bytes={offset}-{end}"}) as response:\n                            report["download_http_status"] = response.status_code\n                            if response.status_code != 206:\n                                raise TransferStopped("download_range_http_failure")\n                            if response.headers.get("Content-Encoding", "identity") != "identity":\n                                raise TransferStopped("unexpected_transfer_encoding")\n                            cr = re.fullmatch(r"bytes (\\d+)-(\\d+)/(\\d+)", response.headers.get("Content-Range", ""))\n                            if not cr or tuple(map(int, cr.groups())) != (offset, end, total):\n                                raise TransferStopped("download_content_range_mismatch")\n                            length = response.headers.get("Content-Length")\n                            if length is not None and int(length) != expected_part:\n                                raise TransferStopped("download_header_size_mismatch")\n                            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):\n                                check_time()\n                                if not chunk:\n                                    continue\n                                received += len(chunk)\n                                if received > expected_part:\n                                    raise TransferStopped("download_range_exceeded_size")\n                                part_digest.update(chunk)\n                                crc = zlib.crc32(chunk, crc)\n                                stream.write(chunk)\n                        if received != expected_part:\n                            raise TransferStopped("download_range_body_size_mismatch")\n                        digest = part_digest\n                        downloaded += received\n                        report["download_range_checks"].append({"start": offset, "end": end, "bytes": received,\n                                                                "crc32_diagnostic": f"{crc:08x}"})\n                        break\n                    except requests.RequestException:\n                        report["download_retry_count"] += 1\n                        if attempt == 2:\n                            raise TransferStopped("download_range_transport_attempt_budget") from None\n        report.update(downloaded_bytes=downloaded, downloaded_sha256=digest.hexdigest())\n        if downloaded != total or digest.hexdigest() != job["expected_sha256"]:\n            raise TransferStopped("download_size_or_sha_mismatch")\n        report["download_sha256_verified"] = True\n        check_time()\n        stage = "upload"\n        offset = 0\n        probe_statuses = []\n        for attempt in range(1, 4):\n            check_time()\n            report["upload_attempts"] = attempt\n            headers = {"Content-Length": str(total - offset)}\n            if offset:\n                headers["Content-Range"] = f"bytes {offset}-{total - 1}/{total}"\n            try:\n                with archive.open("rb", buffering=0) as stream:\n                    stream.seek(offset)\n                    response = session.put(job["put_url"], data=DeadlineReader(stream, check_time), headers=headers,\n                                           allow_redirects=False, timeout=(15, 60))\n                report["upload_http_status"] = response.status_code\n                if response.status_code in (200, 201):\n                    check_time()\n                    report["upload_complete"] = True\n                    report["complete"] = True\n                    break\n                if response.status_code not in (308, 503):\n                    raise TransferStopped("upload_nonresumable_http_failure")\n            except requests.RequestException:\n                # Exception text can contain the scoped URL. Never expose it.\n                report["upload_http_status"] = None\n            check_time()\n            response = session.put(job["put_url"], headers={"Content-Length": "0", "Content-Range": f"bytes */{total}"},\n                                   allow_redirects=False, timeout=(15, 60))\n            check_time()\n            probe_statuses.append(response.status_code)\n            if response.status_code in (200, 201):\n                report.update(upload_complete=True, complete=True, upload_http_status=response.status_code)\n                break\n            if response.status_code != 308:\n                raise TransferStopped("upload_resume_probe_failure")\n            offset = received_offset(response.headers.get("Range"), total)\n        report["resume_probe_http_statuses"] = probe_statuses\n        if not report["complete"]:\n            raise TransferStopped("upload_attempt_budget")\n    except BaseException as error:\n        report.update(complete=False, failure_stage=stage,\n                      failure_code=error.code if isinstance(error, TransferStopped) else "redacted_unexpected_exception",\n                      exception_type=type(error).__name__)\n    finally:\n        session.close()\n        if scratch is not None:\n            try:\n                shutil.rmtree(scratch)\n            except OSError as error:\n                report.update(complete=False, cleanup_exception_type=type(error).__name__)\n        report["own_temp_archive_removed"] = scratch is None or not scratch.exists()\n        report["total_seconds"] = time.monotonic() - start\n        atomic_report(report, output)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--job", type=Path, required=True)\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/transfer_report.json"))\n    parser.add_argument("--scratch-parent", type=Path, default=Path("/kaggle/working"))\n    args = parser.parse_args()\n    # Loading also uses a redacted failure path: malformed private JSON may have\n    # URLs in its decoder message, so do not propagate that message.\n    try:\n        job = json.loads(args.job.read_text())\n    except BaseException as error:\n        result = {"complete": False, "failure_stage": "job_load", "failure_code": "invalid_private_job",\n                  "exception_type": type(error).__name__, "urls_or_upload_tokens_written_to_output": False}\n        atomic_report(result, args.output)\n        print(json.dumps(result), flush=True)\n        raise SystemExit(1)\n    result = transfer(job, args.output, args.scratch_parent)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result["complete"] else 1)\n\n\nif __name__ == "__main__":\n    main()\n'}
JOB_DATASET='alanchoo/rsna-knee-ortho-http-shard-2-read-v2'
PYTHON_CAP=1236
_namespace={'__name__':'http_v2_owned_parent'}
exec(compile(GUARD_SOURCE,'http_v2_owned_parent','exec'),_namespace)
def _prepare_http_child(scratch):
    additions={'RSNA_HTTP_PILOT_START':str(BOOTSTRAP_MONOTONIC)}
    if JOB_DATASET:
        owner,slug=JOB_DATASET.split('/')
        candidates=[Path('/kaggle/input')/'datasets'/owner/slug,Path('/kaggle/input')/slug,Path('/kaggle/input')/owner/slug]
        roots=[p for p in candidates if (p/'cache_read_job.json').is_file() and (p/'http_launch_approval.json').is_file()]
        if not roots:
            for pattern in [f'*/{slug}',f'*/*/{slug}',f'*/*/*/{slug}']:
                roots.extend(p for p in Path('/kaggle/input').glob(pattern) if (p/'cache_read_job.json').is_file() and (p/'http_launch_approval.json').is_file())
        roots=list(dict.fromkeys(roots))
        if len(roots)!=1:raise RuntimeError('Approved owner READ job input is missing or ambiguous')
        additions['RSNA_HTTP_JOB_DIR']=str(roots[0])
    for name,source in TRANSPORT_SOURCES.items():(scratch/name).write_text(source)
    for field,name in [('RSNA_HTTP_DOWNLOADER','download.py'),('RSNA_HTTP_TRANSPORT','transport.py'),
                       ('RSNA_HTTP_APPROVAL','approval.py'),('RSNA_HTTP_FORECAST','forecast.py')]:
        additions[field]=str(scratch/name)
    (scratch/'launcher.py').write_text(CHILD_SOURCE)
    return [sys.executable,str(scratch/'launcher.py')],additions
bootstrap_checkpoint('http_v2_parent_launch')
result=_namespace['guarded_launcher'](_prepare_http_child,BOOTSTRAP_MONOTONIC+PYTHON_CAP,
    '/kaggle/working/http_parent_report.json',scratch_parent='/tmp')
print(json.dumps(result),flush=True)
if not result['complete']:raise RuntimeError('HTTP V2 stage failed; see redacted owned-parent receipt')
report_name='extraction_report.json'
report=json.loads((Path('/kaggle/working/run')/report_name).read_text())
if not report['extraction_complete_for_shards']:
    raise RuntimeError('HTTP V2 stage incomplete')
print(json.dumps(report),flush=True)
bootstrap_checkpoint('complete')
